# 2.2 Data raw quality check

Validaciones de calidad sobre la capa raw, organizadas como el Excel: **hoja → columna** (en el orden de las columnas).
(1) Vista de evaluación, (2) hoja `query`, (3) hoja `catalogo_indicadores`, (4) hoja `catalogo_entornos`, (5) registro de calidad.

Cada validación tiene un `regla_id` = `<hoja>.<columna>.<nn>` (`fila` cuando aplica a la fila completa) y referencia la hipótesis del EDA que la originó.
Estado `aprobado` = decidido en checkpoint; `propuesto` = pendiente de decisión.

Artefactos persistidos: registro de calidad (actividad 1.3) y el detalle `fila_origen × regla_id` que usa `2_3_data_processing` para aplicar tratamientos.

| Artefacto | Ruta |
|---|---|
| Entrada KPIs | `data/1_raw/kpis_raw.parquet` |
| Entrada catálogos | `data/1_raw/catalogo_indicadores_raw.parquet`, `data/1_raw/catalogo_entornos_raw.parquet` |
| Salida registro | `data/2_processed/registro_calidad.csv` |
| Salida filas marcadas | `data/2_processed/filas_marcadas.parquet` |


In [1]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

RAW_ROOT = Path("../../data/1_raw")
PROCESSED_ROOT = Path("../../data/2_processed")
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

KPIS_RAW_PATH = RAW_ROOT / "kpis_raw.parquet"
CAT_IND_RAW_PATH = RAW_ROOT / "catalogo_indicadores_raw.parquet"
CAT_ENT_RAW_PATH = RAW_ROOT / "catalogo_entornos_raw.parquet"
REGISTRO_PATH = PROCESSED_ROOT / "registro_calidad.csv"
FILAS_MARCADAS_PATH = PROCESSED_ROOT / "filas_marcadas.parquet"

# Orden de hojas y columnas tal como vienen en el Excel (nombre raw -> nombre en el Excel)
COLUMNAS_EXCEL = {
    "query": {"fila": "fila", "frente": "Frente", "corte": "Corte", "codigo_equipo": "Codigo_EQU", "nombre_equipo": "EQU",
              "indicador": "Indicador", "resultado": "Resultado", "meta": "Meta", "cumplimiento_reportado": "Cumplimiento"},
    "catalogo_indicadores": {"frente": "Frente", "indicador": "Indicador"},
    "catalogo_entornos": {"codigo_equipo": "Codigo_EQU", "codigo_padre": "Codigo_Padre"},
}
COLUMNAS_NEGOCIO = ["frente", "corte", "codigo_equipo", "nombre_equipo", "indicador", "resultado", "meta", "cumplimiento_reportado"]
LLAVE = ["corte", "codigo_norm", "indicador"]
UMBRAL_COBERTURA = 0.5  # corte con menos de la mitad de los equipos del corte mediano
TOLERANCIA_ABS, TOLERANCIA_REL = 1e-6, 1e-5

# Regla de negocio por indicador inferida en el EDA (1_2_kpi_profiling, H14-H15): fórmula y tope de Cumplimiento
FORMULA_INDICADOR = {
    "Adopción": ("escala", None),
    "Adopción controles ciberseguridad": ("ratio", None),
    "Apps modernizadas": ("ratio", (0, 1.2)),
    "Brecha Ingresos Gastos": ("uno_menos_diferencia", (0, 1.2)),
    "Calidad estructura funcional": ("ratio", None),
    "Cierre de vulnerabilidades": ("ratio", None),
    "Cumplimiento presupuestal": ("ratio", (0, 1.2)),
    "Disponibilidad": ("ratio", None),
    "Ejecución Presupuestal": ("ratio", None),
    "Frente Macro": ("ratio", None),
    "Frente Micro": ("ratio", None),
    "Gestión del Gasto": ("uno_menos_diferencia", None),
    "ICX": ("ratio", None),
    "Impactos a clientes activos por fricciones, quejas y requerimientos": ("escala", None),
    "Incidentes": ("ratio", None),
    "Jaws de Ciclo": ("uno_mas_diferencia", None),
    "Madurez de las aplicaciones": ("ratio", None),
    "Neon": ("ratio", None),
    "Obsolescencia": ("inverso", (0, 1.2)),
    "Percepción": ("escala", None),
    "Pérdida esperada por fraude": ("escalonada", None),
    "Regulatorio": ("ratio", (0, 1)),
    "Talento + Agilidad": ("escala", None),
    "Vulnerabilidades": ("ratio", None),
    "Índice AQR's": ("escalonada", None),
}
FORMULAS = {
    "ratio": lambda r, m: r / m,
    "inverso": lambda r, m: m / r,
    "uno_mas_diferencia": lambda r, m: 1 + (r - m),
    "uno_menos_diferencia": lambda r, m: 1 - (r - m),
    "escala": lambda r, m: r,
}
INDICADORES_ENCUESTA = ["Percepción", "Adopción", "Talento + Agilidad"]
VALOR_COPIADO = 1.014683  # EDA H17: mismo cumplimiento con cientos de resultados distintos

kpis = pd.read_parquet(KPIS_RAW_PATH)
cat_ind = pd.read_parquet(CAT_IND_RAW_PATH)
cat_ent = pd.read_parquet(CAT_ENT_RAW_PATH)
assert set(COLUMNAS_NEGOCIO) <= set(kpis.columns)
assert set(kpis.indicador.unique()) <= set(FORMULA_INDICADOR), "hay indicadores sin regla de negocio definida"
print("kpis_raw | filas:", len(kpis), "| catálogos:", len(cat_ind), "/", len(cat_ent))


kpis_raw | filas: 15188 | catálogos: 13 / 64


## 1. Vista de evaluación

Columnas auxiliares solo para **medir** (no se persisten): código normalizado, indicador sin tildes, padre en el catálogo y cumplimiento esperado según la fórmula del indicador.


In [2]:
def sin_tildes(s):
    return s.str.strip().str.lower().map(
        lambda t: unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode() if pd.notna(t) else t
    )


def cumplimiento_esperado(df):
    esperado = pd.Series(np.nan, index=df.index)
    with np.errstate(divide="ignore", invalid="ignore"):
        for indicador, (formula, tope) in FORMULA_INDICADOR.items():
            if formula not in FORMULAS:
                continue
            m = df.indicador == indicador
            r, meta = df.loc[m, "resultado"].astype(float), df.loc[m, "meta"].astype(float)
            valor = FORMULAS[formula](r, meta)
            valor = valor.where(~(valor.isna() & (r == meta)), np.inf)  # 0/0 = cumplimiento máximo
            esperado[m] = valor if tope is None else valor.clip(*tope)
    return esperado


d = kpis.copy()
d["codigo_norm"] = d.codigo_equipo.str.strip().str.upper().str.replace(r"^([A-Z]{3})0(\d{3})$", r"\g<1>00\2", regex=True)
d["indicador_norm"] = sin_tildes(d.indicador)
d["formula"] = d.indicador.map(lambda i: FORMULA_INDICADOR[i][0])
d["dup_exacto"] = d.duplicated(subset=COLUMNAS_NEGOCIO, keep="first")
base = d[~d.dup_exacto & d.codigo_norm.notna()]
d["llave_repetida"] = base.duplicated(LLAVE, keep=False).reindex(d.index, fill_value=False)
d["respuesta_individual"] = d.llave_repetida & d.corte.eq(202502) & d.indicador.isin(["Percepción", "Adopción"])

padre = d.codigo_norm.map(cat_ent.set_index("codigo_equipo").codigo_padre)
d["esperado"] = cumplimiento_esperado(d)
d["reproducible"] = np.isclose(d.esperado, d.cumplimiento_reportado.astype(float), atol=TOLERANCIA_ABS, rtol=TOLERANCIA_REL)
evaluable = d.formula.isin(list(FORMULAS)) & d[["resultado", "meta", "cumplimiento_reportado"]].notna().all(axis=1)

equipos_por_corte = d[~d.dup_exacto].groupby("corte").codigo_norm.nunique()
cortes_baja_cobertura = equipos_por_corte[equipos_por_corte < UMBRAL_COBERTURA * equipos_por_corte.median()].index
nombre_por_codigo = pd.concat([cat_ent.set_index("codigo_equipo").nombre_equipo,
                               d.dropna(subset=["nombre_equipo"]).groupby("codigo_norm").nombre_equipo.first()])
nombre_por_codigo = nombre_por_codigo[~nombre_por_codigo.index.duplicated()]
catalogo_ind_norm = set(sin_tildes(cat_ind.indicador))
print("cortes con baja cobertura:", list(cortes_baja_cobertura),
      "| filas evaluables por fórmula:", int(evaluable.sum()), "| reproducibles:", int((evaluable & d.reproducible).sum()))


cortes con baja cobertura: [np.int64(202510)] | filas evaluables por fórmula: 13948 | reproducibles: 12906


## 2. Hoja `query`

Columnas en el orden del Excel: `fila` (validaciones de fila completa), `Frente`, `Corte`, `Codigo_EQU`, `EQU`, `Indicador`, `Resultado`, `Meta`, `Cumplimiento`.


In [3]:
def regla(regla_id, validacion, mascara, impacto, tratamiento, justificacion, hipotesis, estado="propuesto"):
    hoja, columna, _ = regla_id.split(".")
    return dict(regla_id=regla_id, hoja=hoja, columna=columna, validacion=validacion, mascara=mascara,
                impacto=impacto, tratamiento=tratamiento, justificacion=justificacion, hipotesis_eda=hipotesis, estado=estado)


no_encuesta = ~d.indicador.isin(INDICADORES_ENCUESTA)
REGLAS_QUERY = [
    # fila
    regla("query.fila.01", "No hay filas duplicadas exactas", d.dup_exacto,
          "Infla conteos y pondera doble a equipos/cortes (202502, 202607).", "excluir",
          "Se conserva la primera ocurrencia; no aporta información nueva.", "H02"),
    regla("query.fila.02", "Percepción/Adopción 202502 vienen una fila por equipo (no por respuesta)", d.respuesta_individual,
          "El equipo pesa según su número de respuestas; rompe el grano mensual.", "corregir",
          "Consolidar al grano equipo-mes con el promedio de respuestas.", "H03", estado="aprobado"),
    regla("query.fila.03", "Corte × Codigo_EQU × Indicador es única (casos no encuesta)", d.llave_repetida & ~d.respuesta_individual,
          "Ambigüedad sobre el valor oficial del mes.", "corregir",
          "Consolidar con el promedio y marcar la llave como consolidada.", "H03", estado="aprobado"),
    # Frente
    regla("query.Frente.01", "Frente existe en el catálogo ('Talento + Agilidad' no existe: nombre 2023-2024)", d.frente.eq("Talento + Agilidad"),
          "El mismo frente aparece con dos nombres y corta la serie histórica.", "corregir",
          "Homologar a 'Modelos de trabajo y Agilidad'.", "H11-H12"),
    regla("query.Frente.02", "Frente escrito sin errores ('Modeos de trabajo y Agilidad')", d.frente.eq("Modeos de trabajo y Agilidad"),
          "Tercer nombre del mismo frente.", "corregir", "Homologar a 'Modelos de trabajo y Agilidad'.", "H11"),
    # Corte
    regla("query.Corte.01", "Cobertura de equipos estable por corte", d.corte.isin(cortes_baja_cobertura),
          "El promedio del corte representa a pocos equipos.", "marcar",
          "Se conserva; los agregados muestran el número de equipos.", "H07"),
    # Codigo_EQU
    regla("query.Codigo_EQU.01", "Codigo_EQU no es nulo", d.codigo_equipo.isna(),
          "La medición no se puede atribuir a ningún equipo.", "excluir", "Sin llave de equipo no hay imputación confiable.", "H01"),
    regla("query.Codigo_EQU.02", "Codigo_EQU con formato AAA00000", d.codigo_equipo.notna() & (d.codigo_equipo != d.codigo_norm),
          "El mismo equipo aparece como dos y no cruza con el catálogo.", "corregir",
          "Mayúsculas y 5 dígitos (Equ00074 -> EQU00074, EQU0024 -> EQU00024).", "H04"),
    regla("query.Codigo_EQU.03", "Codigo_EQU existe en catalogo_entornos", d.codigo_norm.notna() & padre.isna(),
          "Estas filas no se pueden agregar por entorno.", "marcar",
          "Asignar 'Sin entorno'; se mantienen en análisis por equipo y frente.", "H18, H22"),
    regla("query.Codigo_EQU.04", "El padre del equipo es un entorno real (no comodín 'sin Entorno')", padre.str.contains("sin Entorno", na=False),
          "El catálogo reconoce que no tienen entorno.", "marcar", "Se agrupan con 'Sin entorno'.", "H21"),
    regla("query.Codigo_EQU.05", "El padre del equipo es un entorno (ENX) y no una vicepresidencia (VPX)", padre.str.startswith("VPX", na=False),
          "Mezcla niveles jerárquicos en la misma columna.", "marcar",
          "'Sin entorno' en la vista de entornos, conservando la vicepresidencia para su propia lectura.", "H21", estado="aprobado"),
    # EQU
    regla("query.EQU.01", "EQU no es nulo", d.nombre_equipo.isna(),
          "Solo afecta la lectura; el código es la llave.", "corregir",
          "Completar desde el catálogo o desde otras filas del mismo código.", "H01, H08"),
    # Indicador
    regla("query.Indicador.01", "Indicador documentado en catalogo_indicadores", ~d.indicador_norm.isin(catalogo_ind_norm),
          "Sin definición oficial de unidad ni sentido.", "marcar",
          "Se incorpora a dim_indicador con la regla inferida en el EDA como supuesto explícito.", "H09"),
    regla("query.Indicador.02", "Indicador medido en más de un corte (Gestión del Gasto, Jaws de Ciclo: solo 202308)",
          d.indicador.isin(["Gestión del Gasto", "Jaws de Ciclo"]),
          "Sin historia no aporta a tendencias; Gestión del Gasto trae resultados en otra unidad.", "marcar",
          "Se conserva en processed y no entra al puntaje.", "H13, H17"),
    # Resultado
    regla("query.Resultado.01", "Resultado no es nulo cuando hay Cumplimiento", d.resultado.isna() & d.cumplimiento_reportado.notna(),
          "No se puede auditar el cumplimiento.", "aceptar", "El cumplimiento reportado es la cifra oficial.", "H01"),
    # Meta
    regla("query.Meta.01", "Meta positiva en indicadores tipo ratio", d.formula.eq("ratio") & d.meta.le(0),
          "Resultado/Meta no tiene sentido.", "marcar", "Se usa el cumplimiento reportado.", "H05"),
    regla("query.Meta.02", "Meta en una sola unidad por indicador (Índice AQR's mezcla cientos y millones)",
          d.indicador.eq("Índice AQR's") & d.meta.gt(1000),
          "Resultado y Meta no comparables entre periodos.", "marcar", "Se usa el cumplimiento reportado (tabla escalonada).", "H16"),
    regla("query.Meta.03", "Escala de encuesta estable (Percepción/Adopción: meta 10 → 4 → 5)",
          d.indicador.isin(["Percepción", "Adopción"]) & d.meta.isin([10, 4]),
          "Puntajes de distintos años no comparables directamente.", "marcar",
          "La normalización Resultado/Meta absorbe el cambio.", "H16"),
    # Cumplimiento
    regla("query.Cumplimiento.01", "Cumplimiento no es nulo (recuperable con Resultado y Meta)",
          d.cumplimiento_reportado.isna() & d.resultado.notna() & d.meta.notna(),
          "Se perderían mediciones recuperables.", "corregir", "Recalcular con la fórmula del indicador.", "H01"),
    regla("query.Cumplimiento.02", "Cumplimiento no es nulo (sin forma de recalcularlo)",
          d.cumplimiento_reportado.isna() & (d.resultado.isna() | d.meta.isna()),
          "Medición no evaluable.", "excluir", "No hay valor observado ni forma de reconstruirlo.", "H01"),
    regla("query.Cumplimiento.03", "Cumplimiento en la misma escala que el resto (encuestas reportan puntaje bruto)",
          d.indicador.isin(INDICADORES_ENCUESTA),
          "Promediar tal cual favorece 4x a equipos medidos por encuesta.", "corregir",
          "Normalizar como Resultado/Meta.", "H14"),
    regla("query.Cumplimiento.04", f"Cumplimiento no es un valor copiado ({VALOR_COPIADO})",
          np.isclose(d.cumplimiento_reportado.astype(float), VALOR_COPIADO, atol=TOLERANCIA_ABS) & ~d.reproducible,
          "Valor por defecto, no medición: sesga Disponibilidad e Incidentes.", "marcar",
          "Se conserva en processed y no entra al puntaje.", "H17"),
    regla("query.Cumplimiento.05", "Cumplimiento dentro del rango del indicador (Incidentes 202408 = 155.42 en 19 equipos)",
          d.formula.ne("escala") & (d.cumplimiento_reportado.abs() > 3) & d.indicador.ne("Gestión del Gasto"),
          "Un solo valor extremo domina cualquier promedio.", "marcar",
          "Se conserva en processed y no entra al puntaje.", "H15, H17"),
    regla("query.Cumplimiento.06", "Cumplimiento reproducible con la fórmula del indicador (resto de casos)",
          evaluable & ~d.reproducible & ~np.isclose(d.cumplimiento_reportado.astype(float), VALOR_COPIADO, atol=TOLERANCIA_ABS)
          & ~(d.formula.ne("escala") & (d.cumplimiento_reportado.abs() > 3)),
          "No se puede auditar el valor reportado.", "aceptar",
          "Se usa el cumplimiento reportado (cifra oficial) y se marca.", "H17"),
]


## 3. Hoja `catalogo_indicadores`


In [4]:
indicadores_datos = set(d.indicador_norm)
ind_sin_datos = cat_ind[~sin_tildes(cat_ind.indicador).isin(indicadores_datos)].indicador.tolist()
escritura_distinta = sorted(set(cat_ind.indicador) - set(d.indicador) - set(ind_sin_datos))
frente_typo = int(cat_ind.frente.eq("Modeos de trabajo y Agilidad").sum())

REGLAS_CAT_IND = [
    dict(regla_id="catalogo_indicadores.Frente.01", hoja="catalogo_indicadores", columna="Frente",
         validacion="Frente escrito sin errores ('Modeos de trabajo y Agilidad')", filas_afectadas=frente_typo,
         evidencia=f"{frente_typo} filas del catálogo", impacto="El catálogo oficial replica el error de los datos.",
         tratamiento="corregir", justificacion="Homologar a 'Modelos de trabajo y Agilidad'.", hipotesis_eda="H11", estado="propuesto"),
    dict(regla_id="catalogo_indicadores.Indicador.01", hoja="catalogo_indicadores", columna="Indicador",
         validacion="Nombre igual al de los datos", filas_afectadas=len(escritura_distinta),
         evidencia=", ".join(escritura_distinta), impacto="Un cruce exacto falla.",
         tratamiento="corregir", justificacion="Cruzar con nombre normalizado (minúsculas, sin tildes).", hipotesis_eda="H09", estado="propuesto"),
    dict(regla_id="catalogo_indicadores.Indicador.02", hoja="catalogo_indicadores", columna="Indicador",
         validacion="Indicador del catálogo con mediciones", filas_afectadas=len(ind_sin_datos),
         evidencia=", ".join(ind_sin_datos), impacto="Documentación desactualizada o renombrada.",
         tratamiento="aceptar", justificacion="No afecta el cálculo; se reporta para actualizar el catálogo.", hipotesis_eda="H10", estado="propuesto"),
]


## 4. Hoja `catalogo_entornos`


In [5]:
equipos_sin_datos = sorted(set(cat_ent.codigo_equipo) - set(d.codigo_norm.dropna()))
padres_no_entorno = int((~cat_ent.codigo_padre.str.startswith("ENX")).sum())

REGLAS_CAT_ENT = [
    dict(regla_id="catalogo_entornos.Codigo_EQU.01", hoja="catalogo_entornos", columna="Codigo_EQU",
         validacion="Equipo del catálogo con mediciones", filas_afectadas=len(equipos_sin_datos),
         evidencia=", ".join(equipos_sin_datos), impacto="Equipo vigente sin seguimiento.",
         tratamiento="aceptar", justificacion="Se reporta; no altera métricas.", hipotesis_eda="H19", estado="propuesto"),
    dict(regla_id="catalogo_entornos.Codigo_Padre.01", hoja="catalogo_entornos", columna="Codigo_Padre",
         validacion="Padre es un entorno (ENX)", filas_afectadas=padres_no_entorno,
         evidencia=f"{padres_no_entorno} equipos con padre VPX o comodín 'sin Entorno'",
         impacto="La columna mezcla niveles jerárquicos.", tratamiento="marcar",
         justificacion="Agregar tipo_padre (entorno / vicepresidencia / sin entorno).", hipotesis_eda="H21", estado="aprobado"),
]


## 5. Registro de calidad y filas marcadas


In [6]:
def evidencia(mascara):
    n = int(mascara.sum())
    if n == 0:
        return n, "0 filas"
    top = d.loc[mascara, "indicador"].value_counts().head(2)
    return n, f"{n} filas ({n / len(d):.1%}); indicadores con más casos: " + ", ".join(f"{k}: {v}" for k, v in top.items())


filas_registro, filas_marcadas = [], []
for r in REGLAS_QUERY:
    n, texto = evidencia(r["mascara"])
    filas_registro.append({**{k: v for k, v in r.items() if k != "mascara"}, "filas_afectadas": n, "evidencia": texto})
    filas_marcadas.append(pd.DataFrame({"fila_origen": d.loc[r["mascara"], "_fila_origen"].to_numpy(), "regla_id": r["regla_id"]}))
filas_registro += REGLAS_CAT_IND + REGLAS_CAT_ENT

COLUMNAS_REGISTRO = ["regla_id", "hoja", "columna", "validacion", "evidencia", "filas_afectadas", "pct_afectado",
                     "impacto", "tratamiento", "justificacion", "hipotesis_eda", "estado"]
registro = pd.DataFrame(filas_registro)
registro["pct_afectado"] = np.where(registro.hoja == "query", (registro.filas_afectadas / len(d)).round(4), np.nan)

# Orden del Excel: hoja -> columna -> número de regla
orden_hoja = {h: i for i, h in enumerate(COLUMNAS_EXCEL)}
orden_col = {(h, c): i for h, cols in COLUMNAS_EXCEL.items() for i, c in enumerate(cols.values())}
registro = (registro.assign(_h=registro.hoja.map(orden_hoja), _c=[orden_col[(h, c)] for h, c in zip(registro.hoja, registro.columna)])
            .sort_values(["_h", "_c", "regla_id"]).drop(columns=["_h", "_c"])[COLUMNAS_REGISTRO].reset_index(drop=True))
marcadas = pd.concat(filas_marcadas, ignore_index=True).sort_values(["fila_origen", "regla_id"]).reset_index(drop=True)

assert registro.regla_id.is_unique
assert registro.tratamiento.isin(["corregir", "excluir", "marcar", "aceptar"]).all()
assert registro.estado.isin(["aprobado", "propuesto"]).all()

registro.to_csv(REGISTRO_PATH, index=False, encoding="utf-8-sig")
marcadas.to_parquet(FILAS_MARCADAS_PATH, index=False)
print("registro | reglas:", len(registro), "| guardado:", REGISTRO_PATH)
print("filas_marcadas | filas:", len(marcadas), "| filas únicas afectadas:", marcadas.fila_origen.nunique(), "| guardado:", FILAS_MARCADAS_PATH)
display(registro[["regla_id", "validacion", "filas_afectadas", "pct_afectado", "tratamiento", "estado"]])


registro | reglas: 29 | guardado: ..\..\data\2_processed\registro_calidad.csv
filas_marcadas | filas: 35275 | filas únicas afectadas: 12647 | guardado: ..\..\data\2_processed\filas_marcadas.parquet


,regla_id,validacion,filas_afectadas,pct_afectado,tratamiento,estado
0,query.fila.01,No hay filas duplicadas exactas,2182,0.1437,excluir,propuesto
1,query.fila.02,Percepción/Adopción 202502 vienen una fila por equipo (no por resp...,962,0.0633,corregir,aprobado
2,query.fila.03,Corte × Codigo_EQU × Indicador es única (casos no encuesta),303,0.0199,corregir,aprobado
3,query.Frente.01,Frente existe en el catálogo ('Talento + Agilidad' no existe: nomb...,2672,0.1759,corregir,propuesto
4,query.Frente.02,Frente escrito sin errores ('Modeos de trabajo y Agilidad'),417,0.0275,corregir,propuesto
5,query.Corte.01,Cobertura de equipos estable por corte,22,0.0014,marcar,propuesto
6,query.Codigo_EQU.01,Codigo_EQU no es nulo,36,0.0024,excluir,propuesto
7,query.Codigo_EQU.02,Codigo_EQU con formato AAA00000,325,0.0214,corregir,propuesto
8,query.Codigo_EQU.03,Codigo_EQU existe en catalogo_entornos,1473,0.0970,marcar,propuesto
9,query.Codigo_EQU.04,El padre del equipo es un entorno real (no comodín 'sin Entorno'),205,0.0135,marcar,propuesto
